---
title: Linked-Read Alignments
subtitle: Linked-read metrics relating to barcodes and molecules
date: "9999-12-31"
edit_url: null
---

**Linked-read type**: LRTYPE

In [2]:
import altair as alt
from harpy.report.components import colored_boxes, print_html, standard_itable
from harpy.report.utilities import binned_histogram, nxx, last_line
from harpy.report.theme import palette
import itables
import os
import numpy as np
import pandas as pd
from pathlib import Path
import sys
itables.init_notebook_mode(connected=True)

In [3]:
indir = "reports/data/bxstats/"
platform = "haplotagging"

In [30]:
def process_input(infile: str) -> dict:
    samplename = os.path.basename(infile).replace(".bxstats.gz", "")

    tb = pd.read_table(infile, sep = '\t').drop(columns = ["start", "end"])
   
    # Filter conditions - create masks once
    is_valid = tb['molecule'] >= 0
    is_multiplex = is_valid & (tb['reads'] >= 2)
    
    # Calculate metrics using masks
    multiplex_df = tb[is_multiplex]
    #singletons = ((tb['reads'] < 2) & is_valid).sum()
    tot_mol = is_valid.sum()
    tot_valid_reads = tb.loc[is_valid, 'reads'].sum()
    tot_invalid_reads = tb.loc[~is_valid, 'reads'].sum()
    tot_reads = tot_valid_reads + tot_invalid_reads
    # Stats on multiplex molecules
    avg_reads_per_mol = multiplex_df['reads'].mean().round(1)
    #sem_reads_per_mol = multiplex_df['reads'].sem().round(2)
    avg_mol_cov = multiplex_df['coverage_inserts'].mean().round(2)
    #sem_mol_cov = multiplex_df['coverage_inserts'].sem().round(4)
    avg_mol_cov_bp = multiplex_df['coverage_bp'].mean().round(2)
    #sem_mol_cov_bp = multiplex_df['coverage_bp'].sem().round(4)
    _lens = multiplex_df['length_inferred'].sort_values(ascending = False)
    # N-statistics
    n50 = nxx(_lens, 50)
    n75 = nxx(_lens, 75)
    n90 = nxx(_lens, 90)
    
    # Read total unique barcodes from footer
    tot_uniq_bx = int(last_line(infile).split(":")[-1])
    
    return {
        "Sample" : samplename,
        "Total Reads" : tot_reads,
        "Valid" : round(tot_valid_reads/ tot_reads, 4),
        "Unique Molecules" : tot_mol,
        "Linked" : is_multiplex.sum() / tot_mol,
        "Barcodes" : tot_uniq_bx,
        "Barcodes/Molecule" : round(tot_mol / tot_uniq_bx,2),
        "Reads/Molecule" : avg_reads_per_mol,
        #"se_readspermol" : sem_reads_per_mol,
        "Insert Molecule Coverage" : avg_mol_cov,
        #"se_molcov" : sem_mol_cov,
        "BP Molecule Coverage" : avg_mol_cov_bp,
        #"se_molcovbp" : sem_mol_cov_bp,
        "N50" : n50,
        "N75" : n75,
        "N90" : n90
    }

In [39]:
indir = "/home/pdimens/Documents/harpy/Align/bwa/reports/data/bxstats"
agg_cols = ["Sample", "Total Reads", "Valid", "Unique Molecules", "Linked", "Barcodes", "Barcodes/Molecule", "Reads/Molecule", "Insert Molecule Coverage", "BP Molecule Coverage", "N50", "N75", "N90"]
aggregate_df = {}
for i in agg_cols:
    aggregate_df[i] = []

for i in Path(indir).glob("*.bxstats.gz"):
    _sample = process_input(str(i))
    for k,v in _sample.items():
        aggregate_df[k].append(v)

aggregate_df = (
    pd.DataFrame(aggregate_df)
    .sort_values(by = 'Sample')
    .replace(np.nan, 0)
    .reset_index(drop = True)
)

This report aggregates the barcode-specific information from the alignments
that were created using `harpy align`. Detailed information for any one sample
can be found in that sample's individual report.

In [ ]:
if len(aggregate_df) == 0:
    print_html("All input files were empty, no report to show")
    sys.exit(0)

In [37]:
_n50 = (aggregate_df['N50'].replace(0, np.nan).mean()/1000).round()
_n75 = (aggregate_df['N75'].replace(0, np.nan).mean()/1000).round()
_n90 = (aggregate_df['N90'].replace(0, np.nan).mean()/1000).round()

(
    colored_boxes()
    .add(len(aggregate_df), "Samples")
    .conditional(aggregate_df['Linked'].mean().round(3), "Avg Linked", 0.4, as_percent = True)
    .conditional(aggregate_df['Valid'].replace(0, np.nan).mean(), "Avg Valid BX", 0.4, as_percent=True)
    .conditional(aggregate_df['Reads/Molecule'].replace(0, np.nan).mean(), "Avg Reads/Mol", 2)
    .add(_n50, "N50 (kb)", width = 220)
    .add(_n75, "N75 (kb)", width = 220)
    .add(_n90, "N90 (kb)", width = 220)
).render()

## Sample Information

The table below[^cols] is an aggregation of data for each sample based on their `*.bxstats.gz` file.
Every column after `Unique Molecules` ignores singletons in its calculations.

[^cols]:
    | Column | Description |
    |:-------|:------------|
    | `Sample` | name of the sample |
    | `Total Reads`| total number of alignments |
    | `Valid` | proportion of valid barcoded alignments |
    | `Unique Molecules` | the unique DNA molecules as inferred from linked-read barcodes |
    | `Linked` | molecules composed of two or more single/paired-end sequences, in other words, molecules with linked-read information |
    | `Barcodes` | number of unique barcodes, which may differ from unique molecules after deconvolution |
    | `Barcodes/Molecule` | molecule-to-barcode ratio, which helps benchmark deconvolution performance, if performed |
    | `Reads/Molecule` | average number of reads per unique molecule |
    | `Insert Molecule Coverage` | average percent of a molecule that is covered by a read, where coverage includes unsequenced gaps between linked reads |
    | `BP Molecule Coverage` | average percent molecule coverage, where coverage only includes sequences and not the gaps between linked reads |
    | `N50` | N50 of inferred molecules |
    | `N75` | N75 of inferred molecules |
    | `N90` | N90 of inferred molecules |

In [38]:
standard_itable(aggregate_df, "alignment.bx", fixedcols=1)

Loading ITables v2.6.2 from the internet... (need help?)


## Library Performance
### NX metrics

The **NX**[^1] metrics (e.g. **N50**) tend to be useful for understanding centrality for molecule length in genomic contexts.
With your linked-read sequences aligned to a reference genome, you can make inferences about the lengths of the original molecules from which linked-read fragments were derived. These are the distributions of three common NX metrics (N50, N75, N90) across all samples are given below.

[^1]:
    `NX` is the length of the shortest molecule that when you sum the lengths of every molecule larger than it, represents at least **X%** of the total molecules by length. For example, `N50` would be the molecule length at which the sum of all the molecule lengths larger than it would
    represent **50%** of the total molecules by length (sort of like a median).
    
    As an example, imagine we had molecules with lengths [4, 3, 2, 1], making up a total length of 10. The `N50`
    would be the first number (starting from the biggest) that sums up to at least 5 (50% of 10), which is `3`, because `4` + `3` = 7. The `N90` would be the first number that sums up to at least 9 (90% of 10), which is `2` because `4` + `3` + `2` = 9.

:::{note} Understanding Inferred Molecule Lengths
:class: dropdown
Harpy uses the highest and lowest mapping positions of a read cluster sharing the same barcode (incorporating distance-based deconvolution thresholds, if configured to do so) to infer the original molecule length. Given that it's mostly impossible to understand how much of a source molecule the mapped reads actually covered, it would be more appropriate to think of the inferred molecule lengths (and NX measures) to be more like _"at least this long"_, since the reads likely did not originate from the ends of the source DNA molecule. The absolute length of the source DNA molecule would be a more useful diagnostic in troubleshooting/modifying the actual linked-read chemistry, whereas the the inferred molecule length describes what length of it is actually represented in the sequences and thus more useful in downstream/analytical contexts.
:::

In [65]:
selection = alt.selection_point(fields=['Nxx'], bind='legend')

(
    alt.Chart(aggregate_df[['Sample', 'N50', 'N75', 'N90']])
    .transform_fold(
        ['N50', 'N75', 'N90'],
        as_=['Nxx', 'value']
    )
    .transform_density(
        'value',
        groupby=['Nxx'],
        as_=['value', 'density']
    )
    .mark_area(interpolate = "basis")
    .encode(
        x=alt.X('value:Q').axis(title='Length (kb)', labelExpr='datum.value / 1000'),
        y=alt.Y('density:Q', title='Density'),
        color=alt.Color('Nxx:N', title='Metric').scale(domain = ['N50','N75','N90'])
    )
    .transform_filter(selection)
    .add_params(selection)
    .properties(
        title=alt.Title('Nxx Values', subtitle = 'Values derived using non-singleton molecules'),
        usermeta={'embedOptions': {'downloadFileName': f'alignments.nxx'}}   
    )
)

alt.Chart(...)

In [68]:
(
    alt.Chart(aggregate_df[['Sample', 'Reads/Molecule']])
    .transform_density('Reads/Molecule', bandwidth=0.5)
    .mark_area(interpolate = "basis")
    .encode(
        x=alt.X('value:Q').axis(title='Reads Per Molecule'),
        y=alt.Y('density:Q', title='Density'),
    )
    .properties(
        title=alt.Title('Reads Per Molecule', subtitle = 'Values derived using non-singleton molecules'),
        usermeta={'embedOptions': {'downloadFileName': f'alignments.readspermol'}}   
    )
)

alt.Chart(...)

### Reads per molecule
This distribution shows the average number of reads per inferred molecule across your samples.

In [ ]:
if(length(unique(aggregate_df$averagereadspermol)) > 1){
  highchart(height = 400) |>
    hc_chart(type = "areaspline", animation = F, groupPadding = 0.0001) |>
    hc_add_theme(hc_theme_elementary()) |>
    hc_add_series(data = density(aggregate_df$averagereadspermol), color = "#e1a42b",
      name = "Reads per Molecule", marker = list(enabled = FALSE)) |>
    hc_title(text = "Average Reads Per Molecule") |>
    hc_xAxis(title = list(text = "reads per molecule (average)")) |>
    hc_yAxis(title = list(text = "density")) |>
    hc_tooltip(enabled = FALSE) |>
    hc_exporting(enabled = T, filename = "reads_per_molecule",
      buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
    ) |>
    htmlwidgets::onRender(iframe_height(418))
} else {
  IRdisplay::display_html("<p>Not enough unique values for reads per molecule to plot distributions (minimum: 2).<p>")
}

### Valid barcodes
This is a distribution of what percent of total alignments each sample had valid barcodes[^2]:

[^2]:
    Valid barcodes with respect to the conventions of a given linked-read chemistry:
    - **haplotagging**: `AxxCxxBxxDxx` where `xx` is not `00`
    - **stLFR**: `X_Y_Z` where `X`, `Y`, or `Z` is not `0`
    - **TELLseq**: `ATCG` where there is no `N` nucleotide

In [ ]:
hs <- hist(
  aggregate_df$totalvalidpercent,
  breaks = seq(0,100, by = 5),
  plot = F
)
hs$counts <- round(hs$counts / sum(hs$counts) * 100, 4)
hs <- data.frame(val = hs$breaks[-1], freq = hs$counts)

highchart(height = 400) |>
  hc_chart(type = "areaspline", animation = F, groupPadding = 0.0001) |>
  hc_add_theme(hc_theme_elementary()) |>
  hc_add_series(data = hs, color = "#8484bd", name = "Percent Alignments", marker = list(enabled = FALSE),
    hcaes(x = val, y = freq)
  ) |>
  hc_title(text = "Percent of Alignments with Valid BX tags") |>
  hc_xAxis(min = 0, max = 100, title = list(text = "% alignments with valid BX tag")) |>
  hc_yAxis(max = 100, title = list(text = "% samples")) |>
  hc_tooltip(crosshairs = TRUE) |>
  hc_exporting(enabled = T, filename = "percent.valid.dist",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  ) |>
  htmlwidgets::onRender(iframe_height(418))

### Linking proportion
This scatterplot is a diagnostic that shows the relationship between the proportion of linked
reads (reads with linked information, aka not singletons) compared to total reads. Each point is a sample and is colored by the
mean number of reads per molecule for that sample.

In [ ]:
highchart(height = 400) |>
  hc_add_theme(hc_theme_elementary()) |>
  hc_chart(type = "scatter", animation = F, groupPadding = 0.0001) |>
  hc_add_series(data = aggregate_df, name = "Linked-Read Performance",
    hcaes(x = totalreads, y = round(nonsingletons / (nonsingletons + singletons), 2), color = averagereadspermol, name = sample, molper = averagereadspermol)
  ) |>
  hc_title(text = "Linked Reads vs Total Reads") |>
  hc_xAxis(title = list(text = "total reads")) |>
  hc_yAxis(max = 1, min = 0, title = list(text = "proportion of reads with linked-read information")) |>
  hc_tooltip(
    crosshairs = FALSE,
    pointFormat= '<b>{point.name}</b><br>reads: <b>{point.x}</b><br>proportion linked reads: <b>{point.y}</b><br>reads per molecule: <b>{point.molper}</b></br>'
    ) |>
  hc_exporting(enabled = T, filename = "linkedread.performance",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  ) |>
  htmlwidgets::onRender(iframe_height(418))

## Per-Sample Metrics
Below is a series of plots showing per-sample metrics. The meaning of percentages and error bars
are provided in the bottom-left captions of the plots.

### Percent valid barcodes

In [ ]:
highchart(height = 400) |>
  hc_chart(type = "xrange", animation = F, groupPadding = 0.0001) |>
  hc_add_theme(hc_theme_elementary()) |>
  hc_add_series(data = aggregate_df, dataLabels = list(enabled = TRUE),
    hcaes(x = 0, x2 = totalreads, y = rev(0:(n_samples-1)), partialFill = totalvalidpercent/100)
  ) |> 
  hc_xAxis(title = list(text = "Total Alignments")) |> 
  hc_yAxis(title = FALSE, gridLineWidth = 0, categories = rev(aggregate_df$sample)) |>
  hc_caption(text = "Percentage represents the percent of alignments with a valid BX barcode, shown in dark green. Light green represents invalid BX barcodes.") |>
  hc_tooltip(enabled = FALSE) |>
  hc_colors(color = "#95d8a7") |>
  hc_title(text = "Percent Valid Barcode") |>
  hc_exporting(enabled = T, filename = "BX.valid.alignments",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  ) |>
    htmlwidgets::onRender(iframe_height(418))

### Percent linked reads

In [ ]:
highchart(height = 400) |>
  hc_add_theme(hc_theme_elementary()) |>
  hc_chart(type = "xrange", animation = F, groupPadding = 0.0001) |>
  hc_add_series(
    data = aggregate_df,
    name = "mean",
    dataLabels = list(enabled = TRUE),
    hcaes(x = 0, x2 = nonsingletons + singletons, y = rev(0:(n_samples-1)), partialFill = round(nonsingletons / (nonsingletons + singletons), 4))
  ) |>
  hc_xAxis(title = list(text = "Total Alignments with Valid Barcodes")) |> 
  hc_yAxis(title = FALSE, gridLineWidth = 0, categories = rev(aggregate_df$sample)) |>
  hc_caption(text = "Percentage represents the percent of molecules composed of more than 2 single-end or paired-end sequences, shown in dark blue. Light blue represents the singletons.") |>
  hc_tooltip(enabled = FALSE) |>
  hc_colors(color = "#8dc6f5") |>
  hc_title(text = "% Linked Sequences") |>
  hc_exporting(enabled = T, filename = "linked",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  ) |>
    htmlwidgets::onRender(iframe_height(418))

### Reads per Molecule

In [ ]:
err_df <- data.frame(x = 0:(n_samples-1), y = aggregate_df$averagereadspermol, low = aggregate_df$averagereadspermol - aggregate_df$sereadspermol, high = aggregate_df$averagereadspermol + aggregate_df$sereadspermol)
highchart(height = 400) |>
  hc_add_theme(hc_theme_elementary()) |>
  hc_chart(inverted=TRUE, animation = F, pointPadding = 0.0001, groupPadding = 0.0001) |>
  hc_add_series(data = aggregate_df, type = "scatter", name = "mean", hcaes(x = 0:(nrow(aggregate_df)-1), y = averagereadspermol), marker = list(radius = 8), color = "#6d73c2", zIndex = 1) |>
  hc_add_series(data = err_df, type = "errorbar", name = "standard error", linkedTo = ":previous", zIndex = 0, stemColor = "#8186c7", whiskerColor = "#8186c7") |>
  hc_xAxis(title = FALSE, gridLineWidth = 0, categories = aggregate_df$sample) |>
  hc_title(text = "Average Reads Per Molecule") |>
  hc_caption(text = "Error bars show the standard error of the mean of non-singleton molecules") |>
  hc_tooltip(crosshairs = TRUE, pointFormat= '<b>{point.y}</b>') |>
  hc_exporting(enabled = T, filename = "avg.reads.per.molecule",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  ) |>
    htmlwidgets::onRender(iframe_height(418))

### Molecule Coverage

In [ ]:
err_df <- data.frame(x = 0:(n_samples-1), y = aggregate_df$averagemolcov, low = aggregate_df$averagemolcov - aggregate_df$semolcov, high = aggregate_df$averagemolcov + aggregate_df$semolcov)
err_df_bp <- data.frame(x = 0:(n_samples-1), y = aggregate_df$averagemolcovbp, low = aggregate_df$averagemolcovbp - aggregate_df$semolcovbp, high = aggregate_df$averagemolcovbp + aggregate_df$semolcovbp)

highchart(height = 400) |>
  hc_add_theme(hc_theme_elementary()) |>
  hc_chart(inverted=TRUE, animation = F, pointPadding = 0.0001, groupPadding = 0.0001) |>
  hc_add_series(data = aggregate_df, type = "scatter", name = "Inferred Fragments", hcaes(x = 0:(nrow(aggregate_df)-1), y = averagemolcov), marker = list(radius = 8), color = "#df77b5", zIndex = 1) |>
  hc_add_series(data = err_df, type = "errorbar", name = "standard error", linkedTo = ":previous", zIndex = 0, stemColor = "#ef94ca", whiskerColor = "#ef94ca") |>
  hc_add_series(data = aggregate_df, type = "scatter", name = "Aligned Base Pairs", hcaes(x = 0:(nrow(aggregate_df)-1), y = averagemolcovbp), marker = list(radius = 8), color = "#924596", zIndex = 1) |>
  hc_add_series(data = err_df_bp, type = "errorbar", name = "standard error", linkedTo = ":previous", zIndex = 0, stemColor = "#916094", whiskerColor = "#916094") |>
  hc_xAxis(title = FALSE, gridLineWidth = 0, categories = aggregate_df$sample) |>
  hc_title(text = "Average % Molecule Coverage") |>
  hc_caption(text = "Error bars show the standard error of the mean percent coverage of non-singleton molecules") |>
  hc_tooltip(crosshairs = TRUE, pointFormat= '<b>{point.y}</b>') |>
  hc_exporting(enabled = T, filename = "avg.molecule.coverage",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  ) |>
    htmlwidgets::onRender(iframe_height(418))